In [1]:
import json
import re
from dataclasses import asdict, dataclass
from pathlib import Path
import pandas as pd
import pdfplumber

In [2]:
BASE_DIR = Path(r"D:\\Final_GRAG")

PDF_PATH = BASE_DIR / "GRI_standards" /"Sector_standards" / "GRI 11 Oil And Gas Sector 2021.pdf"
METADATA_DIR = BASE_DIR / "metadata"
SECTOR_DIR = METADATA_DIR / "sector_standards" / "gri_11"
SECTOR_DIR.mkdir(parents=True, exist_ok=True)

CSV_OUT = SECTOR_DIR / "gri_11_sector_metadata.csv"
JSON_OUT = SECTOR_DIR / "gri_11_sector_metadata.json"

## Cấu trúc dữ liệu

In [3]:
@dataclass
class SectorDisclosureRow:
    sector_name: str
    topic_id: str
    base_topic_id: str
    topic_name: str
    standards_id: str
    disclosure_id: str

## Regex patterns và tiện ích

In [4]:
TOPIC_RE = re.compile(r"^Topic\s+(11\.\d+)\s+(.+?)\s*$", re.IGNORECASE)
SECTOR_REF_RE = re.compile(r"\b(11\.\d+\.\d+)\b")
DISCLOSURE_RE = re.compile(r"\bDisclosure\s+(\d+-\d+)\s+(.+?)\s*$", re.IGNORECASE)

TRAILING_PAGE_NO_RE = re.compile(r"\s+\d{1,3}\s*$")
INCOMPLETE_TAIL_RE = re.compile(
    r"(,|\b(and|or|of|to|the|with|for|by|on|in|a|an)\b)\s*$",
    re.IGNORECASE,
)


def clean_text(s):
    s = (s or "").replace("\u00a0", " ")
    s = s.replace("\ufb01", "fi").replace("\ufb02", "fl")
    s = s.replace("\ufffd", "'")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _strip_toc_page_no(name):
    """Xóa số trang cuối dòng trong TOC (ví dụ: "GHG emissions 13" → "GHG emissions")."""
    return TRAILING_PAGE_NO_RE.sub("", name).strip()


def iter_pdf_pages_text(pdf_path):
    with pdfplumber.open(str(pdf_path)) as pdf:
        for page_index, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
            yield page_index, lines


def parse_topic_heading(line):
    m = TOPIC_RE.match(line)
    if not m:
        return None
    topic_code = m.group(1).strip()
    topic_name = clean_text(m.group(2)).strip("-\u2013\u2014: ")
    # Xóa số trang TOC trước khi so sánh tên
    topic_name = _strip_toc_page_no(topic_name)
    return topic_code, topic_name

### Lọc tên topic

In [5]:
def _is_suspicious_topic_name(name):
    """Kiểm tra tên topic có phải là fragment/cross-reference bị wrap không."""
    name = clean_text(name)
    lower = name.lower()
    if "see also" in lower:
        return True
    if re.search(r"\btopic\s+11\.\d+\b", lower):
        return True
    if name.endswith(")."):
        return True
    if len(name) > 120:
        return True
    if INCOMPLETE_TAIL_RE.search(name):
        return True
    return False


def _should_update_topic_name(existing, new):
    """
    Kiểm tra xem `new` có phải là tên topic tốt hơn `existing` không.
    Ưu tiên: tên sạch > tên suspicious; cùng loại thì ưu tiên tên dài hơn.
    """
    new = clean_text(new)
    if not new:
        return False

    if not existing:
        return True

    existing_clean = clean_text(existing)
    if existing_clean == new:
        return False

    new_susp = _is_suspicious_topic_name(new)
    existing_susp = _is_suspicious_topic_name(existing_clean)

    # Tên sạch luôn thắng tên suspicious
    if existing_susp and not new_susp:
        return True
    if new_susp and not existing_susp:
        return False

    # Cùng loại → ưu tiên tên dài hơn
    return len(new) > len(existing_clean)

### Xử lý bảng dữ liệu

In [6]:
def _topic_code_from_ref(ref_no):
    parts = (ref_no or "").split(".")
    if len(parts) >= 2:
        return f"{parts[0]}.{parts[1]}"
    return ref_no


def parse_row_text(row_text, *, sector_name, current_standard_id, topic_name_by_code, fallback_topic_name):
    """Xử lý một dòng trong bảng (STANDARD/DISCLOSURE/.../SECTOR STANDARD REF. NO.)."""

    row_text = clean_text(row_text)
    ref_matches = list(SECTOR_REF_RE.finditer(row_text))
    if not ref_matches:
        return None, current_standard_id

    ref_no = ref_matches[-1].group(1)
    row_topic_code = _topic_code_from_ref(ref_no)
    row_topic_name = topic_name_by_code.get(row_topic_code, fallback_topic_name)

    left = row_text[: ref_matches[-1].start()].strip()

    # Lấy standard_id từ dòng hiện tại, hoặc kế thừa từ dòng trước
    standard_id = None
    if left.lower().startswith("gri "):
        m_std = re.match(r"^(GRI\s+\d+)\b", left, flags=re.IGNORECASE)
        if m_std:
            standard_id = clean_text(m_std.group(1)).replace("  ", " ")
            current_standard_id = standard_id
    if not standard_id:
        standard_id = current_standard_id

    if not standard_id:
        return None, current_standard_id

    # Trích xuất disclosure code + tên
    m_dis = DISCLOSURE_RE.search(left)
    if not m_dis:
        # Bỏ qua các mục không phải Disclosure
        return None, current_standard_id

    disclosure_code = clean_text(m_dis.group(1))
    disclosure_name = clean_text(m_dis.group(2))
    disclosure_name = re.sub(r"\s+\d{4}\s*$", "", disclosure_name).strip()

    row = SectorDisclosureRow(
        sector_name=sector_name,
        topic_id=ref_no,
        base_topic_id=row_topic_code,
        topic_name=row_topic_name,
        standards_id=standard_id,
        disclosure_id=disclosure_code,
    )
    return row, current_standard_id

## Trích xuất sector metadata

In [7]:
def extract_gri11_sector_metadata(pdf_path):
    """Trích xuất các hàng sector metadata từ PDF, trước khi dedup."""
    sector_name = pdf_path.stem
    if sector_name.startswith("GRI 11 ") and not sector_name.startswith("GRI 11:"):
        sector_name = "GRI 11: " + sector_name[len("GRI 11 "):].lstrip()

    rows = []
    topic_name_by_code = {}
    current_topic_code = None
    current_topic_name = None
    current_standard_id = None
    buffer = []

    for page_no, lines in iter_pdf_pages_text(pdf_path):
        for raw_line in lines:
            line = clean_text(raw_line)
            if not line:
                continue

            topic = parse_topic_heading(line)
            if topic:
                topic_code, topic_name = topic

                existing = topic_name_by_code.get(topic_code)
                if _should_update_topic_name(existing, topic_name):
                    topic_name_by_code[topic_code] = topic_name

                # Chỉ cập nhật context khi là heading thật, tránh bị reset bởi wrapped references
                if not _is_suspicious_topic_name(topic_name):
                    current_topic_code, current_topic_name = topic_code, topic_name
                    current_standard_id = None
                    buffer.clear()
                    continue

            if not current_topic_code or not current_topic_name:
                continue

            # Buffer các dòng bắt đầu bằng "GRI" hoặc "Disclosure"; flush khi thấy ref no
            if buffer:
                buffer.append(line)
            else:
                if line.lower().startswith("gri ") or line.lower().startswith("disclosure "):
                    buffer = [line]
                else:
                    continue

            if SECTOR_REF_RE.search(line):
                row_text = " ".join(buffer)
                buffer = []
                parsed, current_standard_id = parse_row_text(
                    row_text,
                    sector_name=sector_name,
                    current_standard_id=current_standard_id,
                    topic_name_by_code=topic_name_by_code,
                    fallback_topic_name=current_topic_name,
                )
                if parsed:
                    rows.append(parsed)

        buffer = []

    return rows


def deduplicate_sector_rows(rows):
    """Loại bỏ hàng trùng, giữ nguyên thứ tự."""
    seen = set()
    unique = []
    for r in rows:
        key = (r.topic_id, r.standards_id, r.disclosure_id)
        if key in seen:
            continue
        seen.add(key)
        unique.append(r)

    return unique


def write_outputs(rows, csv_out, json_out):
    rows_list = [asdict(r) for r in rows]

    with open(json_out, "w", encoding="utf-8") as f:
        json.dump(rows_list, f, ensure_ascii=False, indent=2)

    df = pd.DataFrame(rows_list)
    df.to_csv(csv_out, index=False, encoding="utf-8")

    return None

## Chạy pipeline

In [8]:
raw_rows = extract_gri11_sector_metadata(PDF_PATH)
deduped_rows = deduplicate_sector_rows(raw_rows)

print("Extracted rows (raw):", len(raw_rows))
print("Rows after de-duplication:", len(deduped_rows))

# Lưu sector metadata trước khi dedup
write_outputs(raw_rows, CSV_OUT, JSON_OUT)

df_sector = pd.read_csv(CSV_OUT)
df_sector

Extracted rows (raw): 97
Rows after de-duplication: 97


,sector_name,topic_id,base_topic_id,topic_name,standards_id,disclosure_id
0,GRI 11: Oil And Gas Sector 2021,11.1.1,11.10,GHG emissions,GRI 3,3-3
1,GRI 11: Oil And Gas Sector 2021,11.1.2,11.10,GHG emissions,GRI 302,302-1
2,GRI 11: Oil And Gas Sector 2021,11.1.3,11.10,GHG emissions,GRI 302,302-2
3,GRI 11: Oil And Gas Sector 2021,11.1.4,11.10,GHG emissions,GRI 302,302-3
4,GRI 11: Oil And Gas Sector 2021,11.1.5,11.10,GHG emissions,GRI 305,305-1
...,...,...,...,...,...,...
92,GRI 11: Oil And Gas Sector 2021,11.21.5,11.21,Payments to governments,GRI 207,207-2
93,GRI 11: Oil And Gas Sector 2021,11.21.6,11.21,Payments to governments,GRI 207,207-3
94,GRI 11: Oil And Gas Sector 2021,11.21.7,11.21,Payments to governments,GRI 207,207-4
95,GRI 11: Oil And Gas Sector 2021,11.22.1,11.22,Public policy,GRI 3,3-3


In [9]:
df_check = pd.read_csv(CSV_OUT)
print("CSV columns:", df_check.columns.tolist())
df_check.head()

CSV columns: ['sector_name', 'topic_id', 'base_topic_id', 'topic_name', 'standards_id', 'disclosure_id']


,sector_name,topic_id,base_topic_id,topic_name,standards_id,disclosure_id
0,GRI 11: Oil And Gas Sector 2021,11.1.1,11.1,GHG emissions,GRI 3,3-3
1,GRI 11: Oil And Gas Sector 2021,11.1.2,11.1,GHG emissions,GRI 302,302-1
2,GRI 11: Oil And Gas Sector 2021,11.1.3,11.1,GHG emissions,GRI 302,302-2
3,GRI 11: Oil And Gas Sector 2021,11.1.4,11.1,GHG emissions,GRI 302,302-3
4,GRI 11: Oil And Gas Sector 2021,11.1.5,11.1,GHG emissions,GRI 305,305-1


## Gộp với gri_units

In [10]:
GRI_UNITS_CSV = METADATA_DIR / "gri_units" / "gri_units.csv"
GRI_UNITS_JSON = METADATA_DIR / "gri_units" / "gri_units.json"


def load_gri_units():
    """Ưu tiên JSON vì giữ nguyên embeddings."""
    if GRI_UNITS_JSON.exists():
        with open(GRI_UNITS_JSON, "r", encoding="utf-8") as f:
            data = json.load(f)
        return pd.DataFrame(data)

    # Fallback nếu không có JSON
    return pd.read_csv(GRI_UNITS_CSV)


# Master: gri_units
df_units = load_gri_units()


# Sector metadata trước khi dedup
df_sector_raw = pd.read_csv(CSV_OUT)
df_sector = pd.DataFrame(asdict(r) for r in deduplicate_sector_rows(
    SectorDisclosureRow(**row) for row in df_sector_raw.to_dict("records")
))


# Join keys
join_keys = ["standards_id", "disclosure_id"]


# Chuẩn hóa: dùng standard_id làm standards_id
df_units_join = df_units.copy()
df_units_join["standards_id"] = df_units_join["standard_id"]


# df_sector có thể có nhiều hàng per join key → aggregate trước khi merge
def _uniq_join(series):
    vals = [str(v).strip() for v in series.dropna().tolist() if str(v).strip()]
    vals = sorted(set(vals))
    return "; ".join(vals) if vals else pd.NA


sector_value_cols = [c for c in df_sector.columns if c not in join_keys]
df_sector_agg = (
    df_sector.groupby(join_keys, dropna=False, as_index=False)
    .agg({c: _uniq_join for c in sector_value_cols})
)


df_join = df_units_join.merge(
    df_sector_agg,
    how="left",
    on=join_keys,
    validate="m:1",
)


print("gri_units rows:", len(df_units))
print("sector rows (saved raw):", len(df_sector_raw))
print("sector rows (deduped for merge):", len(df_sector))
print("sector rows (agg):", len(df_sector_agg))
print("joined rows:", len(df_join))
print(
    "joined has embeddings:",
    "dense_embedding" in df_join.columns,
    "sparse_embedding" in df_join.columns,
 )
if len(df_join) != len(df_units_join):
    raise ValueError("Join expanded or shrank rows; expected 1:1 with df_units")


df_join.head()

gri_units rows: 787
sector rows (saved raw): 97
sector rows (deduped for merge): 97
sector rows (agg): 68
joined rows: 787
joined has embeddings: False False


,unit_id,standard_id,standard_name,standard_type,disclosure_id,disclosure_name,requirement_id,requirement_text,requirement_type,is_omittable,year,hierarchy_level,parent_requirement,standards_id,sector_name,topic_id,base_topic_id,topic_name
0,GRI101-2024-D101-1-Ra,GRI 101,Biodiversity,topic,101-1,Policies to halt and reverse,a,describe its policies or commitments to halt a...,shall,True,2024,1,None,GRI 101,NaN,NaN,NaN,NaN
1,GRI101-2024-D101-1-Rb,GRI 101,Biodiversity,topic,101-1,Policies to halt and reverse,b,report the extent to which these policies or c...,shall,True,2024,1,None,GRI 101,NaN,NaN,NaN,NaN
2,GRI101-2024-D101-1-Rc,GRI 101,Biodiversity,topic,101-1,Policies to halt and reverse,c,report the goals and targets to halt and rever...,shall,True,2024,1,None,GRI 101,NaN,NaN,NaN,NaN
3,GRI101-2024-D101-2-Ra-i,GRI 101,Biodiversity,topic,101-2,Management of biodiversity impacts,a.i,report how it applies the mitigation hierarchy...,shall,True,2024,2,a,GRI 101,NaN,NaN,NaN,NaN
4,GRI101-2024-D101-2-Ra-ii,GRI 101,Biodiversity,topic,101-2,Management of biodiversity impacts,a.ii,report how it applies the mitigation hierarchy...,shall,True,2024,2,a,GRI 101,NaN,NaN,NaN,NaN


## Lưu kết quả

In [11]:
JOINED_OUT = METADATA_DIR / "gri_units" / "gri_units_with_sector_metadata.csv"
df_join.to_csv(JOINED_OUT, index=False, encoding="utf-8")

In [12]:
JOINED_JSON_OUT = METADATA_DIR / "gri_units" / "gri_units_with_sector_metadata.json"
df_join.to_json(JOINED_JSON_OUT, orient="records", force_ascii=False, indent=2)